# 🔬 Skin Cancer Classification — Transformer Model Comparison
**Models:** DINOv2 · ViT-Base · Swin-Base  
**Framework:** PyTorch + HuggingFace Transformers  
**Platform:** Google Colab (GPU T4/A100)

> Upload your segmented skin lesion image dataset as a ZIP before running.  
> Expected structure: `dataset/train/<class>/`, `dataset/val/<class>/`, `dataset/test/<class>/`


## 1. Install Dependencies

In [ ]:
!pip install -q transformers timm scikit-learn seaborn matplotlib tqdm torchmetrics
print("✅ All packages installed")


## 2. Imports & Config

In [ ]:
import os, time, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchmetrics import Accuracy, F1Score, AUROC, ConfusionMatrix

from transformers import (
    Dinov2Model, ViTForImageClassification,
    SwinForImageClassification
)
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 3. Configuration

In [ ]:
# ─── EDIT THESE ─────────────────────────────────────────────────────
DATASET_PATH   = "./dataset"        # Path to your dataset folder
BATCH_SIZE     = 32                 # Reduce to 16 if VRAM < 8 GB
IMG_SIZE       = 224                # Input resolution
NUM_EPOCHS_HEAD = 5                 # Warm-up epochs (head only)
NUM_EPOCHS_FULL = 15                # Full fine-tuning epochs
BACKBONE_LR    = 1e-5               # Learning rate for backbone
HEAD_LR        = 1e-3               # Learning rate for classification head
WEIGHT_DECAY   = 1e-4
USE_GRAD_CKPT  = False              # Set True if running out of VRAM
MODELS_TO_RUN  = ["dinov2", "vit", "swin"]  # Choose which to train
# ────────────────────────────────────────────────────────────────────

print("⚙️  Config loaded successfully")


## 4. Upload & Extract Dataset

In [ ]:
# Option A: Upload a ZIP file from your local machine
from google.colab import files
import zipfile

print("📁 Upload your dataset ZIP (folder structure: dataset/train/<class>/...)")
uploaded = files.upload()

for fname in uploaded.keys():
    print(f"   Extracting {fname} ...")
    with zipfile.ZipFile(fname, 'r') as zf:
        zf.extractall(".")
    print(f"   ✅ Extracted to current directory")

# Verify structure
for split in ["train", "val", "test"]:
    p = Path(DATASET_PATH) / split
    if p.exists():
        classes = [d.name for d in p.iterdir() if d.is_dir()]
        total   = sum(len(list((p/c).iterdir())) for c in classes)
        print(f"   {split:5s}: {len(classes)} classes, {total} images — {classes}")
    else:
        print(f"   ⚠️  {split} folder not found at {p}")


## 5. Dataset & DataLoaders

In [ ]:
# ── Transforms ──────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Datasets ────────────────────────────────────────────────────────
train_ds = datasets.ImageFolder(f"{DATASET_PATH}/train", transform=train_tf)
val_ds   = datasets.ImageFolder(f"{DATASET_PATH}/val",   transform=val_tf)
test_ds  = datasets.ImageFolder(f"{DATASET_PATH}/test",  transform=val_tf)

NUM_CLASSES = len(train_ds.classes)
CLASS_NAMES  = train_ds.classes
print(f"📊 Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"   Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# ── Class weights for imbalanced data ───────────────────────────────
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_ds.targets)
class_weights = torch.FloatTensor(cw).to(device)
print(f"⚖️  Class weights: {dict(zip(CLASS_NAMES, cw.round(2)))}")

# ── Loaders ─────────────────────────────────────────────────────────
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ── Visualise a sample batch ─────────────────────────────────────────
inv_norm = transforms.Normalize(
    mean=[-m/s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
    std=[1/s for s in IMAGENET_STD]
)
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i, ax in enumerate(axes.flat):
    if i < len(imgs):
        img = inv_norm(imgs[i]).permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img); ax.set_title(CLASS_NAMES[labels[i]], fontsize=7)
    ax.axis("off")
plt.suptitle("Sample Training Batch", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


## 6. Model Definitions

In [ ]:
# ── DINOv2 ───────────────────────────────────────────────────────────
class DINOv2Classifier(nn.Module):
    def __init__(self, num_classes, use_grad_ckpt=False):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained("facebook/dinov2-base")
        if use_grad_ckpt:
            self.backbone.encoder.gradient_checkpointing = True
        hidden = self.backbone.config.hidden_size  # 768
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        out = self.backbone(pixel_values=x)
        cls = out.last_hidden_state[:, 0]   # CLS token
        return self.classifier(cls)

# ── ViT ──────────────────────────────────────────────────────────────
def build_vit(num_classes):
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224",
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
    )
    return model

# ── Swin Transformer ─────────────────────────────────────────────────
def build_swin(num_classes):
    model = SwinForImageClassification.from_pretrained(
        "microsoft/swin-base-patch4-window7-224",
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
    )
    return model

# ── Factory ──────────────────────────────────────────────────────────
def build_model(name, num_classes):
    if name == "dinov2":
        return DINOv2Classifier(num_classes, use_grad_ckpt=USE_GRAD_CKPT)
    elif name == "vit":
        return build_vit(num_classes)
    elif name == "swin":
        return build_swin(num_classes)
    raise ValueError(f"Unknown model: {name}")

# ── Parameter count helper ────────────────────────────────────────────
def count_params(model):
    total   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print("✅ Model builders defined")


## 7. Training Utilities

In [ ]:
def get_logits(model, images):
    """Unified forward pass — handles HuggingFace and custom models.""""
    out = model(images)
    return out.logits if hasattr(out, "logits") else out

def freeze_backbone(model, model_name):
    if model_name == "dinov2":
        for p in model.backbone.parameters(): p.requires_grad = False
    elif model_name == "vit":
        for p in model.vit.parameters(): p.requires_grad = False
    elif model_name == "swin":
        for p in model.swin.parameters(): p.requires_grad = False

def unfreeze_backbone(model):
    for p in model.parameters(): p.requires_grad = True

def make_optimizer(model, model_name, stage):
    if stage == "head":
        params = [p for p in model.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    # Stage 2: differential LR
    if model_name == "dinov2":
        backbone_params = list(model.backbone.parameters())
        head_params     = list(model.classifier.parameters())
    elif model_name == "vit":
        backbone_params = list(model.vit.parameters())
        head_params     = list(model.classifier.parameters())
    elif model_name == "swin":
        backbone_params = list(model.swin.parameters())
        head_params     = list(model.classifier.parameters())
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params,     "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, lbls in tqdm(loader, leave=False, desc="  train"):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        logits = get_logits(model, imgs)
        loss   = criterion(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        logits     = get_logits(model, imgs)
        total_loss += criterion(logits, lbls).item() * imgs.size(0)
        correct    += (logits.argmax(1) == lbls).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

print("✅ Training utilities ready")


## 8. Train All Models

In [ ]:
criterion    = nn.CrossEntropyLoss(weight=class_weights)
all_history  = {}   # model_name → {train_loss, val_loss, train_acc, val_acc}
best_models  = {}   # model_name → state_dict of best val checkpoint

for model_name in MODELS_TO_RUN:
    print(f"\n{'='*60}")
    print(f"  Training: {model_name.upper()}")
    print(f"{'='*60}")

    model = build_model(model_name, NUM_CLASSES).to(device)
    total, _ = count_params(model)
    print(f"  Parameters: {total/1e6:.1f}M")

    history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}
    best_val_acc = 0.0

    # ── Stage 1: Warm-up (head only) ─────────────────────────────────
    print(f"\n  Stage 1 — Head warm-up ({NUM_EPOCHS_HEAD} epochs)")
    freeze_backbone(model, model_name)
    optimizer = make_optimizer(model, model_name, "head")

    for epoch in range(1, NUM_EPOCHS_HEAD + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0
        history["train_loss"].append(tr_loss);  history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc);    history["val_acc"].append(vl_acc)
        print(f"  Ep {epoch:02d}/{NUM_EPOCHS_HEAD} | "
              f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | "
              f"vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} | {elapsed:.1f}s")

    # ── Stage 2: Full fine-tuning ─────────────────────────────────────
    print(f"\n  Stage 2 — Full fine-tuning ({NUM_EPOCHS_FULL} epochs)")
    unfreeze_backbone(model)
    optimizer = make_optimizer(model, model_name, "full")
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS_FULL, eta_min=1e-7
    )

    for epoch in range(1, NUM_EPOCHS_FULL + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        scheduler.step()
        elapsed = time.time() - t0
        history["train_loss"].append(tr_loss);  history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc);    history["val_acc"].append(vl_acc)
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_models[model_name] = copy.deepcopy(model.state_dict())
            ckpt_path = f"best_{model_name}.pth"
            torch.save(best_models[model_name], ckpt_path)
            print(f"  Ep {epoch:02d}/{NUM_EPOCHS_FULL} | "
                  f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | "
                  f"vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} | "
                  f"✅ BEST saved → {ckpt_path} | {elapsed:.1f}s")
        else:
            print(f"  Ep {epoch:02d}/{NUM_EPOCHS_FULL} | "
                  f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | "
                  f"vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} | {elapsed:.1f}s")

    all_history[model_name] = history
    # Free VRAM before loading next model
    del model; torch.cuda.empty_cache()

print("\n✅ Training complete for all models!")


## 9. Training Curves

In [ ]:
COLOR_MAP = {"dinov2": "#01696f", "vit": "#006494", "swin": "#7a39bb"}
LABELS    = {"dinov2": "DINOv2-Base", "vit": "ViT-Base/16", "swin": "Swin-Base"}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Training Curves — All Models", fontsize=15, fontweight="bold")

for model_name, hist in all_history.items():
    c  = COLOR_MAP[model_name]
    lbl = LABELS[model_name]
    ep = range(1, len(hist["train_acc"]) + 1)
    axes[0].plot(ep, [a*100 for a in hist["train_acc"]], c=c, ls="--",  alpha=0.6, lw=1.5)
    axes[0].plot(ep, [a*100 for a in hist["val_acc"]],   c=c, ls="-",   lw=2, label=lbl)
    axes[1].plot(ep, hist["train_loss"],  c=c, ls="--", alpha=0.6, lw=1.5)
    axes[1].plot(ep, hist["val_loss"],    c=c, ls="-",  lw=2, label=lbl)

for ax, title, ylabel in zip(axes,
    ["Accuracy", "Loss"],
    ["Accuracy (%)", "Cross-Entropy Loss"]):
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)
    # shade warm-up region
    ax.axvspan(0, NUM_EPOCHS_HEAD + 0.5, alpha=0.05, color="orange",
               label="Warm-up phase")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: training_curves.png")


## 10. Test Set Evaluation

In [ ]:
@torch.no_grad()
def full_evaluation(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, lbls in tqdm(loader, desc="  eval", leave=False):
        imgs = imgs.to(device)
        logits = get_logits(model, imgs)
        probs  = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu())
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(lbls.tolist())
    return (np.array(all_labels),
            np.array(all_preds),
            torch.cat(all_probs).numpy())

test_results = {}

for model_name in MODELS_TO_RUN:
    print(f"\n  Evaluating {LABELS[model_name]} on test set ...")
    model = build_model(model_name, NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(f"best_{model_name}.pth", map_location=device))

    labels, preds, probs = full_evaluation(model, test_loader)

    acc = (preds == labels).mean()
    report = classification_report(labels, preds,
                                   target_names=CLASS_NAMES, output_dict=True)
    macro_f1  = report["macro avg"]["f1-score"]
    macro_pre = report["macro avg"]["precision"]
    macro_rec = report["macro avg"]["recall"]

    test_results[model_name] = {
        "labels": labels, "preds": preds, "probs": probs,
        "accuracy": acc, "macro_f1": macro_f1,
        "macro_precision": macro_pre, "macro_recall": macro_rec,
        "report": report
    }

    print(f"  Accuracy:  {acc*100:.2f}%")
    print(f"  Macro F1:  {macro_f1:.4f}")
    print(f"  Macro Pre: {macro_pre:.4f}  Macro Rec: {macro_rec:.4f}")
    del model; torch.cuda.empty_cache()

print("\n✅ All models evaluated!")


## 11. Results Comparison Table

In [ ]:
rows = []
for name in MODELS_TO_RUN:
    r = test_results[name]
    best_acc_hist = max(all_history[name]["val_acc"]) * 100
    rows.append({
        "Model":         LABELS[name],
        "Test Acc (%)":  round(r["accuracy"]*100, 2),
        "Best Val Acc (%)": round(best_acc_hist, 2),
        "Macro F1":      round(r["macro_f1"], 4),
        "Macro Prec.":   round(r["macro_precision"], 4),
        "Macro Recall":  round(r["macro_recall"], 4),
    })

df_results = pd.DataFrame(rows).set_index("Model")
df_results = df_results.sort_values("Test Acc (%)", ascending=False)

# ── Styled display ────────────────────────────────────────────────────
print("\n" + "="*70)
print(df_results.to_string())
print("="*70)
print(f"\n🏆 Best model: {df_results['Test Acc (%)'].idxmax()}")

df_results.to_csv("model_comparison_results.csv")
print("\n✅ Saved: model_comparison_results.csv")


## 12. Confusion Matrices

In [ ]:
n_models = len(MODELS_TO_RUN)
fig, axes = plt.subplots(1, n_models, figsize=(8 * n_models, 6))
if n_models == 1: axes = [axes]

for ax, name in zip(axes, MODELS_TO_RUN):
    r = test_results[name]
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(r["labels"], r["preds"])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, vmin=0, vmax=1,
                annot_kws={"size": 9})
    ax.set_title(f"{LABELS[name]}\nTest Acc: {r['accuracy']*100:.2f}%",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("True Label", fontsize=10)
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)

plt.suptitle("Normalised Confusion Matrices", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: confusion_matrices.png")


## 13. Per-Class F1 Score Comparison

In [ ]:
per_class = {}
for name in MODELS_TO_RUN:
    rep = test_results[name]["report"]
    per_class[LABELS[name]] = [rep[c]["f1-score"] for c in CLASS_NAMES]

df_f1 = pd.DataFrame(per_class, index=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(max(10, len(CLASS_NAMES)*1.5), 5))
x  = np.arange(len(CLASS_NAMES))
w  = 0.8 / len(MODELS_TO_RUN)
colors = [COLOR_MAP[n] for n in MODELS_TO_RUN]

for i, (col, c) in enumerate(zip(df_f1.columns, colors)):
    bars = ax.bar(x + i*w - w*(len(MODELS_TO_RUN)-1)/2,
                  df_f1[col], width=w*0.9, label=col, color=c, alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_ylim(0, 1.08); ax.set_ylabel("F1 Score"); ax.set_xlabel("Class")
ax.set_title("Per-Class F1 Score — Model Comparison", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.axhline(0.8, ls="--", c="gray", lw=1, alpha=0.5, label="0.8 target")
plt.tight_layout()
plt.savefig("per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: per_class_f1.png")


## 14. ROC-AUC Curves (Macro OvR)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

fig, axes = plt.subplots(1, len(MODELS_TO_RUN), figsize=(7*len(MODELS_TO_RUN), 6))
if len(MODELS_TO_RUN) == 1: axes = [axes]

for ax, name in zip(axes, MODELS_TO_RUN):
    r   = test_results[name]
    lb  = label_binarize(r["labels"], classes=range(NUM_CLASSES))
    fpr_all, tpr_all, roc_auc_all = {}, {}, {}

    for i, cname in enumerate(CLASS_NAMES):
        fpr_all[i], tpr_all[i], _ = roc_curve(lb[:, i], r["probs"][:, i])
        roc_auc_all[i] = auc(fpr_all[i], tpr_all[i])

    # Macro AUC
    all_fpr = np.unique(np.concatenate([fpr_all[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
    mean_tpr /= NUM_CLASSES
    macro_auc = auc(all_fpr, mean_tpr)

    cmap = plt.cm.get_cmap("tab10", NUM_CLASSES)
    for i, cname in enumerate(CLASS_NAMES):
        ax.plot(fpr_all[i], tpr_all[i], lw=1.5, c=cmap(i),
                label=f"{cname} (AUC={roc_auc_all[i]:.2f})", alpha=0.75)

    ax.plot(all_fpr, mean_tpr, "k--", lw=2.5,
            label=f"Macro-avg (AUC={macro_auc:.3f})")
    ax.plot([0,1],[0,1], "gray", lw=1, ls=":")
    ax.set_title(f"{LABELS[name]}", fontsize=12, fontweight="bold")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(fontsize=7, loc="lower right"); ax.grid(alpha=0.3)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)

    test_results[name]["macro_auc"] = macro_auc

plt.suptitle("ROC-AUC Curves (One-vs-Rest)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("roc_auc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: roc_auc_curves.png")


## 15. Grad-CAM Visualisation (Best Model)

In [ ]:
# Install grad-cam library
!pip install -q grad-cam

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Load best model
best_name  = df_results["Test Acc (%)"].idxmax()
best_key   = [k for k, v in LABELS.items() if v == best_name][0]
print(f"🔎 Generating Grad-CAM for: {best_name}")

model = build_model(best_key, NUM_CLASSES).to(device)
model.load_state_dict(torch.load(f"best_{best_key}.pth", map_location=device))
model.eval()

# Target layer (last transformer block)
if best_key == "dinov2":
    target_layers = [model.backbone.encoder.layer[-1].norm1]
elif best_key == "vit":
    target_layers = [model.vit.encoder.layer[-1].layernorm_before]
elif best_key == "swin":
    target_layers = [model.swin.encoder.layers[-1].blocks[-1].layernorm_before]

imgs, lbls = next(iter(test_loader))
imgs_dev   = imgs[:8].to(device)

def reshape_transform_vit(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    return result.transpose(2, 3).transpose(1, 2)

cam = GradCAM(model=model, target_layers=target_layers,
              reshape_transform=reshape_transform_vit if best_key != "swin" else None)

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i in range(8):
    img_tensor = imgs_dev[i:i+1]
    grayscale_cam = cam(input_tensor=img_tensor,
                        targets=[ClassifierOutputTarget(lbls[i].item())])
    rgb_img = inv_norm(imgs[i]).permute(1,2,0).clamp(0,1).numpy()
    vis     = show_cam_on_image(rgb_img.astype(np.float32), grayscale_cam[0], use_rgb=True)
    axes[0, i].imshow(rgb_img);   axes[0, i].set_title(CLASS_NAMES[lbls[i]], fontsize=7)
    axes[1, i].imshow(vis);       axes[1, i].set_title("Grad-CAM", fontsize=7)
    axes[0, i].axis("off");       axes[1, i].axis("off")

plt.suptitle(f"Grad-CAM — {best_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("gradcam_visualization.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: gradcam_visualization.png")
del model; torch.cuda.empty_cache()


## 16. Final Summary & Download Results

In [ ]:
print("\n" + "="*70)
print("  FINAL RESULTS SUMMARY")
print("="*70)

# Build final table with AUC
rows_final = []
for name in MODELS_TO_RUN:
    r = test_results[name]
    rows_final.append({
        "Model":          LABELS[name],
        "Test Acc (%)":   round(r["accuracy"]*100, 2),
        "Macro F1":       round(r["macro_f1"], 4),
        "Macro Prec.":    round(r["macro_precision"], 4),
        "Macro Recall":   round(r["macro_recall"], 4),
        "Macro ROC-AUC":  round(r.get("macro_auc", 0), 4),
    })

df_final = pd.DataFrame(rows_final).set_index("Model").sort_values("Test Acc (%)", ascending=False)
print(df_final.to_string())
print("="*70)
print(f"\n🏆 Best model: {df_final['Test Acc (%)'].idxmax()}")
print(f"   Test Accuracy:   {df_final['Test Acc (%)'].max():.2f}%")
print(f"   Macro F1 Score:  {df_final.loc[df_final['Test Acc (%)'].idxmax(), 'Macro F1']:.4f}")
print(f"   Macro ROC-AUC:   {df_final.loc[df_final['Test Acc (%)'].idxmax(), 'Macro ROC-AUC']:.4f}")

df_final.to_csv("final_model_comparison.csv")

# ── Download all outputs ──────────────────────────────────────────────
from google.colab import files
for fname in [
    "final_model_comparison.csv",
    "training_curves.png",
    "confusion_matrices.png",
    "per_class_f1.png",
    "roc_auc_curves.png",
    "gradcam_visualization.png",
] + [f"best_{n}.pth" for n in MODELS_TO_RUN]:
    if os.path.exists(fname):
        files.download(fname)
        print(f"⬇️  Downloaded: {fname}")


## 17. Inference on a Single Image

In [ ]:
from PIL import Image

def predict_image(image_path, model_name, top_k=3):
    model = build_model(model_name, NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(f"best_{model_name}.pth", map_location=device))
    model.eval()

    img = Image.open(image_path).convert("RGB")
    tensor = val_tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = get_logits(model, tensor)
        probs  = F.softmax(logits, dim=1)[0]

    top_probs, top_idxs = probs.topk(top_k)
    print(f"\n📸 {image_path} → {LABELS[model_name]}")
    print(f"{'Rank':<6}{'Class':<25}{'Confidence':>12}")
    print("-"*45)
    for rank, (prob, idx) in enumerate(zip(top_probs, top_idxs), 1):
        bar = "█" * int(prob.item() * 30)
        print(f"  {rank}  {CLASS_NAMES[idx.item()]:<23} {prob.item()*100:>8.2f}%  {bar}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img); ax1.set_title("Input Image"); ax1.axis("off")
    colors = [COLOR_MAP.get(model_name, "#333")] * top_k
    colors[0] = COLOR_MAP.get(model_name, "#01696f")
    bars = ax2.barh([CLASS_NAMES[i] for i in top_idxs.cpu().tolist()],
                    [p.item()*100 for p in top_probs],
                    color=colors, alpha=0.85)
    ax2.set_xlabel("Confidence (%)"); ax2.set_xlim(0, 100)
    ax2.set_title(f"Top-{top_k} Predictions")
    for bar, p in zip(bars, top_probs):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f"{p.item()*100:.1f}%", va="center")
    plt.tight_layout(); plt.show()
    del model; return CLASS_NAMES[top_idxs[0].item()], top_probs[0].item()

# Usage:
# predict_image("path/to/your/image.jpg", "dinov2")
print("✅ Inference function ready — call predict_image('image.jpg', 'dinov2')")
